# 🧪 vLLM Tensor-Parallel Test Server (Qwen2.5-Coder-14B, Unity-tuned) — Kaggle Edition

**This is a separate, experimental variant** of the `transformers`-based Unified Generate+Repair
Server. It exists to test one specific thing: **true tensor-parallel model sharding via vLLM**,
instead of either of the two approaches used elsewhere:

- The *original* notebook used `device_map="auto"` — this is **pipeline sharding**: GPU 0 holds
  layers 1-20, GPU 1 holds layers 21-40, and a request runs through them sequentially. Only one
  GPU is ever actively computing at a time; the other idles until the pipeline reaches it.
- The *dual-GPU* notebook that followed used **two full independent model copies**, one per GPU
  (data parallelism) — real parallel throughput across *different* requests, but each GPU still
  holds the *entire* model's weights alone.

**This notebook does neither.** It uses vLLM's `tensor_parallel_size=2`, which shards the
model's weight matrices themselves evenly across both GPUs — a 12GB model becomes ~6GB per card
— and for a *single* request, both GPUs compute simultaneously, exchanging activations over PCIe
after each layer. This is genuine tensor parallelism, not an approximation of it.

It reproduces every feature of the original server: the Unity-tuned LoRA adapter (via vLLM's
native LoRA support, no separate PEFT model loading needed), the full
planning → generation → bootstrap → readme → zip → validation pipeline, the `/repair` endpoint,
the unified Flask job server with OOM-aware retries, the live dashboard, and the ngrok tunnel.

**Before running:** this needs a fresh kernel with nothing executed yet. Creating CUDA context in
this notebook process *before* vLLM spins up its tensor-parallel worker processes can break the
worker spawn — if this hangs or errors oddly on first load, `Restart Session` and `Run All` from
a clean state rather than re-running cells individually.


## Step 0 — Environment check

In [ ]:
import subprocess, sys
print(subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total,memory.used", "--format=csv"],
                      capture_output=True, text=True).stdout)
import psutil
print(f"System RAM: {psutil.virtual_memory().total / 1e9:.1f} GB")


## Step 1 — Install dependencies

In [ ]:
import subprocess, sys

def pip_install(pkgs):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-input"] + pkgs, check=True)

# vLLM bundles its own compatible torch/transformers -- installing it fresh (rather than reusing
# whatever the base Kaggle image ships) avoids version-mismatch errors between vLLM and torch,
# a very common source of import-time crashes with this combination.
pip_install(["-U", "vllm"])

# CHANGED: vLLM does NOT bundle bitsandbytes itself -- quantization="bitsandbytes" in the
# model-loading cell needs the actual package present inside every worker process (each GPU
# runs its own worker under tensor parallelism), or loading fails with "ModuleNotFoundError:
# No module named 'bitsandbytes'" right as weights are being sharded onto the GPUs. This was
# the missing piece -- everything else (NCCL, both TP workers, the LoRA adapter match) was
# already initializing correctly in the log before this killed it.
pip_install(["bitsandbytes>=0.48.1"])

pip_install(["-U", "peft"])       # lightweight: only used here to read adapter_config.json
                                    # (base-model check, LoRA rank) -- never used to load weights.
pip_install(["flask", "pyngrok"])
print("\u2705 Dependencies installed.")


## Step 2 — Working directory setup

In [ ]:
import os, time, json as _json

WORKDIR = "/kaggle/working/llm_project_gen_vllm"
LOG_DIR = os.path.join(WORKDIR, "logs")
JOBS_DIR = os.path.join(WORKDIR, "jobs")
for d in (WORKDIR, LOG_DIR, JOBS_DIR):
    os.makedirs(d, exist_ok=True)

print(f"Working directory: {WORKDIR}")
print(f"Each /generate job gets its own isolated subfolder under {JOBS_DIR}.")


## Step 3 — Load Qwen2.5-Coder-14B via vLLM, sharded across both GPUs, with the LoRA adapter

This is the whole point of this notebook: `tensor_parallel_size` tells vLLM to split the model's
weight matrices evenly across that many GPUs (2 T4s here). For a 4-bit-quantized ~9-11GB model,
that means each GPU holds roughly half -- ~5GB -- of the weights, and **both GPUs compute on
every single request together**, not one after the other.

On top of that, vLLM pre-allocates a chunk of each GPU's memory (`gpu_memory_utilization`, set
below) as a KV-cache pool for fast continuous batching. That means each GPU's total memory usage
will be noticeably higher than "half the weights" alone -- that's intentional and is what makes
vLLM fast, not a leak or a misconfiguration.

The LoRA adapter is attached natively through vLLM's `enable_lora` + `LoRARequest` mechanism --
no separate PEFT-wrapped model object needed. The adapter's own `base_model_name_or_path` is
still checked against `MODEL_ID` before attaching, same safety check as the other notebooks.


In [ ]:
import torch, glob
# NOTE: torch.cuda.device_count() alone does not create a CUDA context in this process (it's a
# lightweight cudaGetDeviceCount call) -- safe to call before vLLM spins up its own TP workers.
# Anything that actually touches a CUDA tensor here (e.g. `torch.zeros(1).cuda()`) would NOT be
# safe to do before the LLM() call below -- avoid that if you're customizing this cell.

MODEL_ID = "Qwen/Qwen2.5-Coder-14B-Instruct"

n_gpus = torch.cuda.device_count()
print(f"{n_gpus} GPU(s) visible.")
TENSOR_PARALLEL_SIZE = max(n_gpus, 1)

from transformers import AutoTokenizer
print("Loading tokenizer (for chat templating and spec-length truncation only -- generation "
      "itself goes through vLLM, not this tokenizer's model)...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# --- Adapter detection + compatibility check (config only, no weight loading) ------------------
from peft import PeftConfig
from vllm.lora.request import LoRARequest

_adapter_candidates = [
    os.path.dirname(p) for p in glob.glob("/kaggle/input/**/adapter_config.json", recursive=True)
]
if not _adapter_candidates and os.path.isdir("/kaggle/input"):
    print("    (No adapter_config.json found anywhere under /kaggle/input. Full tree, for debugging:)")
    for _root, _dirs, _files in os.walk("/kaggle/input"):
        for _f in _files:
            print(f"      {os.path.join(_root, _f)}")

USING_TUNED_MODEL = False
ADAPTER_LORA_REQUEST = None
_adapter_max_rank = None

if _adapter_candidates:
    ADAPTER_DIR = _adapter_candidates[0]
    _peft_cfg = PeftConfig.from_pretrained(ADAPTER_DIR)
    _adapter_base = _peft_cfg.base_model_name_or_path or ""
    _base_tail = MODEL_ID.split("/")[-1].lower()
    _adapter_tail = _adapter_base.split("/")[-1].lower().replace("-bnb-4bit", "")
    if _base_tail.replace("-instruct", "") not in _adapter_tail and _adapter_tail not in _base_tail:
        print(f"\u274c Adapter/base MISMATCH -- refusing to attach.")
        print(f"    Adapter was trained on: {_adapter_base}")
        print(f"    This notebook is loading: {MODEL_ID}")
    else:
        _adapter_max_rank = max(int(_peft_cfg.r), 8)  # vLLM needs max_lora_rank >= adapter's own rank
        ADAPTER_LORA_REQUEST = LoRARequest("unity_adapter", 1, ADAPTER_DIR)
        USING_TUNED_MODEL = True
        print(f"\u2705 Adapter base ({_adapter_base}) matches -- will attach via vLLM LoRA "
              f"(rank={_peft_cfg.r}): {ADAPTER_DIR}")
else:
    print("\u26a0\ufe0f  No LoRA adapter dataset found -- running the plain base model.")
    print("    Attach it via: Add Input -> search for your adapter dataset -> select it, then "
          "Run -> Restart Session and Run All (Kaggle only mounts /kaggle/input/ at kernel start).")

if USING_TUNED_MODEL:
    print("\nUsing the Unity-tuned model (adapter applied per-request via LoRARequest) for all generation below.")
else:
    print("\n\u26a0\ufe0f Using the base (non-tuned) model for all generation below.")

# --- Load the model, sharded across TENSOR_PARALLEL_SIZE GPUs ----------------------------------
from vllm import LLM, SamplingParams

_llm_kwargs = dict(
    model=MODEL_ID,
    quantization="bitsandbytes",
    load_format="bitsandbytes",   # loads the HF repo's weights and quantizes to 4-bit on the fly,
                                    # same nf4 4-bit footprint as the other notebooks' BitsAndBytesConfig
    dtype="float16",               # T4 (sm75) has no real bf16 support -- float16 throughout
    tensor_parallel_size=TENSOR_PARALLEL_SIZE,
    gpu_memory_utilization=0.85,   # fraction of EACH GPU vLLM is allowed to claim (weights + KV
                                    # cache pool). Lower this (e.g. 0.75) if you see startup OOMs;
                                    # raise it if there's headroom and you want a bigger KV cache.
    max_model_len=8192,            # ceiling on prompt+output tokens for any single request --
                                    # matches this pipeline's PLANNING_SPEC_MAX_CHARS-scale prompts
                                    # plus generation headroom; raise if your specs are larger and
                                    # you have the KV-cache budget for it.
    enforce_eager=True,             # CHANGED: T4s (compute capability 7.5) don't support
                                    # bitsandbytes' fused 4-bit matmul kernel, so bnb falls back to
                                    # _dequant_linear_fallback -- it reconstructs the FULL
                                    # dequantized weight matrix on the fly for every matmul. That's
                                    # fine for normal inference, but vLLM's CUDA graph capture step
                                    # (which pre-captures execution graphs across ~100 different
                                    # batch sizes, 1 up to 512, to speed up later inference) piles
                                    # that extra transient memory on top of an already-reserved KV
                                    # cache budget -- and blew past it here specifically during
                                    # graph capture, not during normal generation. enforce_eager
                                    # skips CUDA graph capture (and torch.compile) entirely and
                                    # runs each forward pass eagerly instead -- somewhat slower per
                                    # token, but sidesteps this exact OOM. Worth revisiting if you
                                    # ever move to GPUs with native int4 tensor core support
                                    # (Ampere/cc 8.0+), where the fused kernel path applies and
                                    # this restriction isn't needed.
    trust_remote_code=True,
)
if USING_TUNED_MODEL:
    _llm_kwargs["enable_lora"] = True
    _llm_kwargs["max_lora_rank"] = _adapter_max_rank

print(f"Loading {MODEL_ID} via vLLM, tensor_parallel_size={TENSOR_PARALLEL_SIZE}...")
print("(This can take a few minutes on first run -- downloading + quantizing + sharding weights "
      "across GPUs, plus vLLM's own startup profiling pass.)")
llm = LLM(**_llm_kwargs)
print(f"\u2705 Model loaded and sharded across {TENSOR_PARALLEL_SIZE} GPU(s) via tensor parallelism.")

for _i in range(n_gpus):
    _free_b, _total_b = torch.cuda.mem_get_info(_i)
    print(f"    GPU {_i} memory after load: {(_total_b - _free_b) / 1e9:.2f} GB used / "
          f"{_total_b / 1e9:.2f} GB total ({_free_b / 1e9:.2f} GB free). Includes vLLM's "
          f"pre-allocated KV-cache pool on top of the sharded weights -- expected, not a leak.")


## Step 4 — Generation helpers: uncapped length, auto-continues instead of truncating

In [ ]:
MAX_NEW_TOKENS_PER_CALL = 2048  # smaller per call, more continuations -- lower peak
                                 # memory per call while still reaching the same total length
MAX_CONTINUATIONS = 8          # hard safety ceiling: up to ~18K generated tokens per file

GENERATION_MODE = "deterministic"  # "deterministic" or "creative"
PLANNING_TEMPERATURE = 0.1  # planning stays low-temperature regardless of mode -- consistent,
                             # parseable JSON structure matters more here than variety
FILE_GEN_TEMPERATURE = {"deterministic": 0.15, "creative": 0.6}[GENERATION_MODE]
print(f"Generation mode: {GENERATION_MODE} (file temperature = {FILE_GEN_TEMPERATURE})")

MAX_FIX_ATTEMPTS = 2  # Feature 3: how many times to regenerate a file that fails validation
                       # before giving up and leaving it flagged in the final report

import gc, re, collections
GENERATION_LOG = os.path.join(LOG_DIR, "generation.log")

def _log(msg):
    ts = time.strftime("%H:%M:%S")
    line = f"[{ts}] {msg}"
    print(line)
    with open(GENERATION_LOG, "a") as f:
        f.write(line + "\n")


# --- Live status + work-log tracking, shown on the ngrok dashboard ("/") ---------------------
# _ACTIVITY holds a single human-readable sentence describing whatever the pipeline is doing
# right this second (e.g. "Generating Assets/.../SetupRunner.cs..."). _STEP_LOG is a rolling
# history of completed units of work (one planning pass, one generated/repaired file, README,
# zip, validation...), each with how big it was and how long it took, newest first.
_ACTIVITY = {"text": "Idle -- waiting for a job.", "since": time.time()}
_STEP_LOG = collections.deque(maxlen=300)
_CURRENT_JOB_ID = None


def _set_activity(text):
    """Updates what the dashboard shows under 'Currently doing'."""
    _ACTIVITY["text"] = text
    _ACTIVITY["since"] = time.time()
    _log(f"[activity] {text}")


def _record_step(name, size_label, seconds, phase="generate"):
    """Logs one completed unit of work into the work-log table on the dashboard.
    size_label is a pre-formatted string like '12,353 chars' or '2.34 MB'."""
    _STEP_LOG.appendleft({
        "job_id": _CURRENT_JOB_ID,
        "phase": phase,
        "name": name,
        "size_label": size_label,
        "seconds": seconds,
        "ts": time.time(),
    })
    _log(f"[step] {phase}/{name}: {size_label}, {seconds:.1f}s")


import threading, contextlib

@contextlib.contextmanager
def _heartbeat(label, interval=15):
    """Ticks _ACTIVITY (and the log) every `interval` seconds for as long as a blocking
    llm.generate() call is in flight, so a genuinely slow call is visibly distinct from a
    hung one on the dashboard -- instead of the same line sitting there for minutes with no
    way to tell "still working" from "stuck". T4 + enforce_eager + bitsandbytes' dequant
    fallback + tensor-parallel NCCL sync (see cell 8) makes single calls legitimately slow;
    this doesn't speed anything up, it just makes that visible instead of silent."""
    start = time.time()
    stop_event = threading.Event()

    def _tick():
        while not stop_event.wait(interval):
            _set_activity(f"{label} -- still running ({time.time() - start:.0f}s elapsed, "
                           f"no output yet -- slow is expected on T4+eager+4-bit, not necessarily stuck).")

    t = threading.Thread(target=_tick, daemon=True)
    t.start()
    try:
        yield
    finally:
        stop_event.set()
        t.join(timeout=1)


def _sampling_params(max_tokens, temperature):
    do_sample = temperature > 0
    return SamplingParams(
        max_tokens=max_tokens,
        temperature=max(temperature, 0.01) if do_sample else 0.0,
        top_p=0.9 if do_sample else 1.0,
        repetition_penalty=1.15,
        # CHANGED: repetition_penalty alone wasn't enough -- a real run collapsed into a short
        # token loop ("locklocklocklock...") for the ENTIRE 4000-token planning budget, burning
        # ~20 minutes on garbage. frequency_penalty and presence_penalty are additive logit
        # penalties (a different, often more effective mechanism against exact short loops than
        # repetition_penalty's multiplicative scaling) -- stacking both gives much stronger
        # protection against this specific degenerate-repeat failure mode.
        frequency_penalty=0.4,
        presence_penalty=0.3,
    )


def _looks_degenerate(text, min_chunk=3, max_chunk=12, min_repeats=20):
    """Cheap detector for the 'model fell into a short repeating loop' failure (e.g.
    'locklocklocklock...' instead of real content) -- checks whether some short substring near
    the end of the output repeats an excessive number of times back-to-back. Only checks the
    tail (last 4000 chars): once this collapse starts, it persists to the end of the generation,
    so scanning the whole string isn't necessary."""
    if len(text) < min_chunk * min_repeats:
        return False
    tail = text[-4000:]
    for chunk_len in range(min_chunk, max_chunk + 1):
        chunk = tail[-chunk_len:]
        if not chunk.strip():
            continue
        if (chunk * min_repeats) in tail:
            return True
    return False


def generate_long(system_prompt, user_prompt, max_new_tokens=MAX_NEW_TOKENS_PER_CALL,
                   max_continuations=MAX_CONTINUATIONS, temperature=0.2, seed_text=""):
    """Generate a response with no artificial length cap -- keeps continuing past each call's
    max_new_tokens ceiling until the model produces a natural stop, or the safety ceiling hits.
    Runs through vLLM's tensor-parallel `llm` -- both GPUs work on this one request together.

    seed_text: if provided, resumes from a partial result that already hit its length cap
    elsewhere."""
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": user_prompt},
    ]
    full_output = ""
    start_attempt = 0
    if seed_text:
        full_output = seed_text
        messages.append({"role": "assistant", "content": seed_text})
        messages.append({"role": "user", "content":
            "Continue exactly where you left off. Do not repeat anything already written, do not "
            "restate the file or add any preamble -- just continue the raw content seamlessly."})
        start_attempt = 1

    for attempt in range(start_attempt, max_continuations + 1):
        prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        sampling_params = _sampling_params(max_new_tokens, temperature)

        try:
            with _heartbeat(f"Generating (call {attempt + 1}/{max_continuations + 1})"):
                outputs = llm.generate([prompt], sampling_params, lora_request=ADAPTER_LORA_REQUEST,
                                        use_tqdm=False)
        except Exception as e:
            # vLLM's failure modes differ from raw transformers -- it pre-allocates a fixed KV
            # cache pool at startup (gpu_memory_utilization above) rather than allocating per
            # call, so a request that can't be scheduled raises here instead of a mid-generate
            # torch.cuda.OutOfMemoryError. One retry with a smaller max_tokens covers the common
            # case (a request right at the edge of max_model_len / the KV cache budget).
            _log(f"  vLLM generate() failed ({type(e).__name__}: {e}) -- retrying once with "
                 f"half max_tokens...")
            gc.collect()
            torch.cuda.empty_cache()
            sampling_params = _sampling_params(max(256, max_new_tokens // 2), temperature)
            with _heartbeat(f"Generating (call {attempt + 1}, retry after failure)"):
                outputs = llm.generate([prompt], sampling_params, lora_request=ADAPTER_LORA_REQUEST,
                                        use_tqdm=False)

        result = outputs[0].outputs[0]
        chunk = result.text
        full_output += chunk
        hit_length_cap = (result.finish_reason == "length")  # vLLM tells us directly -- no need
                                                                # to compare token counts by hand

        if not hit_length_cap:
            _log(f"Generation finished naturally after {attempt + 1} call(s), "
                 f"{len(full_output)} chars total.")
            break

        if attempt == max_continuations:
            _log(f"\u26a0\ufe0f Hit the {max_continuations}-continuation safety ceiling -- "
                 f"stopping with {len(full_output)} chars (may be incomplete).")
            break

        _log(f"Call {attempt + 1} hit its length cap -- continuing...")
        messages.append({"role": "assistant", "content": chunk})
        messages.append({"role": "user", "content":
            "Continue exactly where you left off. Do not repeat anything already written, do not "
            "restate the file or add any preamble -- just continue the raw content seamlessly."})

    return full_output


BATCH_SIZE = 2  # how many files to hand vLLM per round. CHANGED from the other notebooks: this is
                 # no longer a manual GPU-splitting concern -- vLLM's own continuous batching
                 # scheduler decides how to interleave concurrent requests across the (already
                 # tensor-parallel-sharded) model internally. Raise this freely; vLLM queues and
                 # batches whatever you hand it in one generate() call far better than any manual
                 # thread/lock scheme would.
CURRENT_BATCH_SIZE = BATCH_SIZE


def generate_batch(system_prompt, user_prompts, max_new_tokens=MAX_NEW_TOKENS_PER_CALL, temperature=0.15):
    """Generate for several prompts in ONE vLLM call. Unlike the other notebooks, there's no
    manual GPU dispatch here -- vLLM's scheduler handles batching multiple requests together
    internally, on top of the tensor-parallel shard that's already splitting each individual
    request's compute across both GPUs. One call, both GPUs, full batching."""
    prompts = []
    for up in user_prompts:
        messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": up}]
        prompts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))

    sampling_params = _sampling_params(max_new_tokens, temperature)
    with _heartbeat(f"Generating a batch of {len(prompts)}"):
        outputs = llm.generate(prompts, sampling_params, lora_request=ADAPTER_LORA_REQUEST, use_tqdm=False)

    results = []
    for o in outputs:
        text = o.outputs[0].text
        hit_cap = (o.outputs[0].finish_reason == "length")
        results.append((text, hit_cap))
    return results


## Step 5 — The generation pipeline (planning → generation → bootstrap → readme → zip → validation)

Same pipeline as the other notebooks, unchanged in structure -- it only ever calls `generate_long`
/ `generate_batch`, so it doesn't need to know or care that those now run through vLLM's
tensor-parallel engine underneath. `run_full_pipeline(spec_text, job_dir)` is still the single
entry point the job server calls for every `/generate` request.


In [ ]:
import shutil, zipfile

TARGET_UNITY_VERSION = "6000.0.35f1"  # <-- change this to match your installed Editor version

PLANNING_SYSTEM_PROMPT = """You are a senior Unity/C# engineer turning a written technical \
specification into a concrete file plan for a Unity project.

The specification you're given may include a document that enumerates the EXACT scripts required \
at this stage -- often called something like "ScriptsIndex.md", a scripts table, or a file list, \
usually organized by folder (Data/, Systems/, Gameplay/, UI/, Tests/, etc.) with one row per \
script giving its exact filename and a description. If a document like that is present, IT IS \
AUTHORITATIVE AND COMPLETE: you must plan exactly one file per script it lists, using that exact \
filename, placed under a path that mirrors its folder (e.g. a script listed under "Systems/" as \
"InventorySystem.cs" becomes "Assets/_Project/Scripts/Systems/InventorySystem.cs"). Do not skip \
any row, do not merge two rows into one file, and do not invent extra scripts beyond what that \
document lists. If the specification instead only gives you prose (no such table), use your own \
judgment to identify every distinct system, data type, and UI panel prose describes, and plan one \
file per one of those.

There is NO CAP on how many files you plan. Plan every file the specification actually calls for \
-- if an authoritative script list names 40 scripts, plan 40 files, not a trimmed-down subset. \
Under-planning (silently dropping files to keep the list short) is a worse failure than a long \
response, because every dropped file is a script the generated project will be missing entirely.

Only plan files you can write correctly as plain text with no Unity Editor available:
- C# scripts (MonoBehaviours and plain classes)
- JSON data files (for content that would normally be ScriptableObject assets -- a human will \
import these into real ScriptableObjects later; do NOT plan .asset files yourself)
- Packages/manifest.json
- Plain text project settings notes
- Exactly one HUMAN_SETUP.md as the last file, explaining what a human must do in the Unity \
Editor to finish wiring this into a working project (scenes, prefabs, importing the JSON data \
into real ScriptableObjects, NavMesh baking, etc.)

Do NOT plan .unity, .prefab, or .asset files -- those need Unity's Editor to keep GUID/fileID \
references valid and will be corrupted if hand-written as raw text.

Do NOT plan a top-level README.md. One is generated automatically after your files are written, \
summarizing everything in this plan -- planning your own would just be overwritten and wastes a \
file you could have spent on an actual script. The only project-level document you plan by hand \
is HUMAN_SETUP.md.

======================================================================================
MANDATORY, NO EXCEPTIONS -- you MUST include exactly one file at:
    "Assets/Editor/ProjectSetup/SetupRunner.cs"
This is the single most important file in the whole plan, and the plan is considered INVALID \
without it. It is an Editor-only script (wrap the whole class in #if UNITY_EDITOR) with a \
[MenuItem("Tools/Run Project Setup")] method that uses Unity's real Editor APIs -- AssetDatabase, \
EditorSceneManager, PrefabUtility, GameObject, AddComponent<T>() -- to programmatically build the \
scene and attach EVERY MonoBehaviour-derived script planned above to an appropriate GameObject, \
instead of a human doing it by hand. If you plan 20 MonoBehaviours, SetupRunner must create or \
reuse GameObjects for all 20 of them (grouped sensibly -- e.g. a Player object, a Managers object \
holding singleton-style systems, a Canvas holding UI panels, fixtures like shelves/checkout \
counters/spawn points as their own objects) -- not just one or two illustrative examples. This is \
what turns a folder of generated CODE into an actually-wired-up, openable scene.

Example of the SHAPE a SetupRunner takes (yours must wire up THIS project's actual planned \
scripts and scene needs -- attaching every one of them, not a copy of this short illustration):
```csharp
#if UNITY_EDITOR
using UnityEditor;
using UnityEditor.SceneManagement;
using UnityEngine;

public static class SetupRunner
{
    [MenuItem("Tools/Run Project Setup")]
    public static void Run()
    {
        var scene = EditorSceneManager.NewScene(NewSceneSetup.DefaultGameObjects);

        var player = new GameObject("Player");
        player.AddComponent<PlayerController>();
        player.AddComponent<CharacterController>();

        var managers = new GameObject("Managers");
        managers.AddComponent<DayCycleManager>();
        managers.AddComponent<CustomerSpawner>();
        managers.AddComponent<SaveLoadService>();

        var canvasGO = new GameObject("Canvas", typeof(Canvas));
        canvasGO.AddComponent<HUDController>();
        canvasGO.AddComponent<CheckoutUI>();

        // ...continue for every remaining planned MonoBehaviour: shelves, checkout counter,
        // customer spawn points, delivery pallet, etc. -- one AddComponent<T>() per script.

        EditorSceneManager.SaveScene(scene, "Assets/_Project/Scenes/Main.unity");
        Debug.Log("Project setup complete.");
    }
}
#endif
```
======================================================================================

CRITICAL path rule: every "path" value MUST start with the literal prefix "Assets/" or
"Packages/" -- for example "Assets/_Project/Scripts/Data/ProductData.cs", never just
"_Project/Scripts/Data/ProductData.cs" or "Scripts/Data/ProductData.cs". A Unity project
is only valid if its actual content lives inside a real Assets/ folder at the project
root -- omitting that prefix will produce a broken, unopenable project.

Output ONLY a JSON array, no other text, no markdown code fences. Each element:
{"path": "Assets/... or Packages/...", "type": "csharp|json|manifest|text|readme", \
"description": "what this file must contain, specific enough to write it correctly in isolation"}

Keep each "description" to ONE short, plain-language sentence (max ~30 words) describing WHAT \
the file does -- e.g. "ScriptableObject holding a product's name, price, and shelf category." \
Do NOT write actual C# code, attributes, field declarations, or syntax in the description -- that \
level of detail belongs in the file itself (generation below writes the real code from this \
description), not in the plan. Verbose code-like descriptions are the main reason planning runs \
out of its token budget before finishing the file list -- keep descriptions short specifically so \
you have room to list every file.
"""

PLANNING_SPEC_MAX_CHARS = 12000  # vLLM's KV cache is pre-allocated as a fixed pool at startup
# (gpu_memory_utilization above), not allocated fresh per call the way raw transformers does --
# so a single large prompt is less likely to hit a hard per-call OOM here than in the
# transformers-based notebooks. Still kept well under max_model_len (8192 tokens) with room for
# the system prompt and response, rather than pushing right up against that ceiling.

GENERATION_SYSTEM_PROMPT = """You are a senior Unity/C# engineer. Write ONE complete, real, \
compiling file -- never a stub, never a placeholder, never a TODO, never truncated. Take as much \
space as the file genuinely needs to be correct and complete; do not artificially shorten it. \
Output ONLY the raw file content -- no markdown code fences, no explanation before or after, no \
commentary. If writing C#, include using statements, full method bodies, and real logic matching \
the description exactly."""

SETUPRUNNER_PATH = "Assets/Editor/ProjectSetup/SetupRunner.cs"


def _try_parse_json(text):
    try:
        return _json.loads(text), None
    except _json.JSONDecodeError as e:
        return None, e


_GUARANTEED_SECTIONS = ["ScriptsIndex.md", "ProjectVersion.txt"]


def _smart_truncate_spec(spec_text, max_chars):
    """Splits the combined spec back into its per-file "=== filename ===" sections (as built by
    _combine_docs_zip), guarantees _GUARANTEED_SECTIONS survive in full, and fills the remaining
    budget with everything else in its existing (already priority-ordered) sequence."""
    if len(spec_text) <= max_chars:
        return spec_text, False

    parts = spec_text.split("\n\n\n")
    sections = {}
    order = []
    for part in parts:
        if part.startswith("=== ") and " ===" in part[4:]:
            fname = part[4:part.index(" ===", 4)]
        else:
            fname = f"(untitled section {len(order)})"
        sections[fname] = part
        order.append(fname)

    guaranteed = [f for f in _GUARANTEED_SECTIONS if f in sections]
    guaranteed_text = "\n\n\n".join(sections[f] for f in guaranteed)
    remaining_budget = max_chars - len(guaranteed_text) - (len("\n\n\n") if guaranteed else 0)

    others_in_order = [f for f in order if f not in guaranteed]
    kept = {}
    used = 0
    for fname in others_in_order:
        sec = sections[fname]
        sep_cost = len("\n\n\n") if used > 0 or guaranteed else 0
        if used + sep_cost + len(sec) <= remaining_budget:
            kept[fname] = sec
            used += sep_cost + len(sec)
        else:
            space_left = remaining_budget - used - sep_cost
            if space_left > 300:
                kept[fname] = sec[:space_left] + "\n[...section truncated for length...]"
            break

    final_order = [f for f in order if f in guaranteed or f in kept]
    result = "\n\n\n".join(sections[f] if f in guaranteed else kept[f] for f in final_order)
    return result, True


def _expected_script_count(spec_text):
    m = re.search(r"[Tt]otal scripts[^:\n]*:\s*(\d+)", spec_text)
    return int(m.group(1)) if m else None


def _run_one_planning_attempt(spec_for_planning, extra_instruction=None, temperature=None):
    prompt = f"Specification:\n\n{spec_for_planning}\n\nProduce the file plan JSON now."
    if extra_instruction:
        prompt += f"\n\nIMPORTANT: {extra_instruction}"

    plan_raw = generate_long(PLANNING_SYSTEM_PROMPT, prompt,
                              max_new_tokens=4000, max_continuations=0,
                              temperature=PLANNING_TEMPERATURE if temperature is None else temperature)

    # CHANGED: check for a repetition-loop collapse BEFORE spending time on JSON repair/salvage --
    # a real run produced 15,600 chars of "locklocklocklock..." here, and salvage logic has no
    # chance of recovering anything useful from that. Failing fast with a clear message (instead
    # of the generic "Expecting value: line 1 column 1" JSON error) also makes run_planning's
    # temperature-escalation retry below kick in immediately rather than after a wasted parse
    # attempt.
    if _looks_degenerate(plan_raw):
        _log(f"\u274c Planning output collapsed into a short repeated-token loop instead of JSON "
             f"(e.g. a several-character chunk repeated dozens of times) -- a sampling failure, "
             f"not a parsing one. First 200 chars: {plan_raw[:200]!r}")
        raise ValueError("Planning generation collapsed into a repetitive loop (degenerate output)")

    plan_clean = plan_raw.strip()
    if plan_clean.startswith("```"):
        plan_clean = plan_clean.split("```")[1]
        if plan_clean.startswith("json"):
            plan_clean = plan_clean[4:]
    plan_clean = plan_clean.strip()

    file_plan, err = _try_parse_json(plan_clean)

    if file_plan is None:
        repaired = re.sub(r',(\s*[\]\}])', r'\1', plan_clean)
        file_plan, err = _try_parse_json(repaired)
        if file_plan is None:
            repaired2 = repaired.replace("'", '"')
            file_plan, err = _try_parse_json(repaired2)

    if file_plan is None:
        last_complete = plan_clean.rfind("},")
        if last_complete != -1:
            salvaged = plan_clean[:last_complete + 1].rstrip().rstrip(",") + "\n]"
            file_plan, salvage_err = _try_parse_json(salvaged)
            if file_plan is not None:
                _log(f"\u26a0\ufe0f Plan was truncated mid-file -- salvaged {len(file_plan)} "
                     f"complete file entries before the cutoff and dropped the incomplete last one.")

    if file_plan is None:
        _log(f"\u274c Planning JSON still invalid after repair attempts: {err}")
        _log(f"Raw model output (first 2000 chars): {plan_raw[:2000]}")
        raise ValueError(f"Planning failed to produce valid JSON: {err}")

    return file_plan


def run_planning(spec_text):
    """Step 1 of the pipeline: turn a spec into a concrete file plan. Returns FILE_PLAN
    (a list of {"path", "type", "description"} dicts)."""
    _set_activity(f"Planning the file structure from the spec ({len(spec_text)} chars)...")
    _plan_step_start = time.time()
    spec_for_planning, _was_truncated = _smart_truncate_spec(spec_text, PLANNING_SPEC_MAX_CHARS)
    if _was_truncated:
        _log(f"\u26a0\ufe0f spec is {len(spec_text)} chars -- truncated to fit "
             f"{PLANNING_SPEC_MAX_CHARS} for the planning prompt. ScriptsIndex.md and "
             f"ProjectVersion.txt (if present) are always kept in FULL regardless of length.")

    expected_count = _expected_script_count(spec_text)
    if expected_count:
        _log(f"Spec declares {expected_count} scripts at this stage (found a 'Total scripts' "
             f"line) -- will sanity-check the plan against this.")

    _log(f"Starting planning pass... ({'Unity-tuned' if USING_TUNED_MODEL else 'BASE (untuned)'} model)")
    # CHANGED: retrying planning with the EXACT SAME low temperature (0.1) after a repetition-loop
    # collapse is likely to just collapse the same way again -- each attempt costs ~20+ min on
    # this hardware, so a blind identical retry is very expensive for basically no chance of a
    # different outcome. Escalate temperature on each local retry (still catches genuine JSON
    # formatting failures too, not just collapses) before giving up and letting the job-level
    # retry (which re-plans from scratch) take over.
    _planning_temp_tiers = [PLANNING_TEMPERATURE, 0.3, 0.5]
    file_plan = None
    _last_planning_err = None
    for _temp_i, _temp in enumerate(_planning_temp_tiers):
        if _temp_i > 0:
            _log(f"\u26a0\ufe0f Planning attempt {_temp_i} failed ({_last_planning_err}) -- "
                 f"retrying with a higher temperature ({_temp}) to break out of a possible "
                 f"repetition loop...")
        try:
            file_plan = _run_one_planning_attempt(spec_for_planning, temperature=_temp)
            break
        except ValueError as e:
            _last_planning_err = e
            continue
    if file_plan is None:
        _log(f"\u274c Planning failed at every temperature tier ({_planning_temp_tiers}) -- "
             f"giving up on this job (the job-level auto-retry will re-plan from scratch).")
        raise _last_planning_err

    if expected_count and len(file_plan) < expected_count * 0.8:
        _log(f"\u26a0\ufe0f Planning only produced {len(file_plan)} files against {expected_count} "
             f"expected -- retrying planning once with an explicit reminder of the shortfall.")
        planned_names = ", ".join(os.path.basename(e["path"]) for e in file_plan)
        retry_file_plan = _run_one_planning_attempt(
            spec_for_planning,
            extra_instruction=(
                f"Your previous attempt only planned {len(file_plan)} files, but the "
                f"specification's script index calls for {expected_count} scripts. You already "
                f"planned: {planned_names}. Plan the FULL list this time -- every script the "
                f"specification's index names, not a subset. Keep descriptions short (a single "
                f"sentence each) specifically so you have room to list every one."
            ),
        )
        if len(retry_file_plan) > len(file_plan):
            _log(f"\u2705 Retry produced {len(retry_file_plan)} files (up from {len(file_plan)}) -- using it.")
            file_plan = retry_file_plan
        else:
            _log(f"\u26a0\ufe0f Retry didn't improve -- keeping the original plan and continuing anyway.")

    if not any(e["path"] == SETUPRUNNER_PATH for e in file_plan):
        _log("\u26a0\ufe0f Model omitted SetupRunner.cs despite the mandatory instruction -- "
             "injecting it into the plan directly rather than leaving it out.")
        file_plan.append({
            "path": SETUPRUNNER_PATH,
            "type": "csharp",
            "description": (
                "Editor-only script (#if UNITY_EDITOR) with a [MenuItem] method that builds the "
                "main scene and attaches EVERY generated MonoBehaviour script planned above to an "
                "appropriate GameObject using AddComponent<T>(), then saves the scene."
            ),
        })

    _log(f"\u2705 Planned {len(file_plan)} files:")
    for entry in file_plan:
        _log(f"  - {entry['path']}  ({entry['type']})")

    seen_lower = {}
    collisions = []
    for entry in file_plan:
        key = entry["path"].lower()
        if key in seen_lower:
            collisions.append((seen_lower[key], entry["path"]))
        else:
            seen_lower[key] = entry["path"]
    if collisions:
        _log(f"\u26a0\ufe0f {len(collisions)} case-insensitive filename collision(s) found in the plan:")
        for a, b in collisions:
            _log(f"    {a!r}  <->  {b!r}")
    else:
        _log("\u2705 No filename collisions in the plan.")

    _record_step("Planning", f"{len(_json.dumps(file_plan)):,} chars",
                 time.time() - _plan_step_start, phase="plan")

    return file_plan


def _check_balance(content):
    problems = []
    for open_ch, close_ch in [("{", "}"), ("(", ")"), ("[", "]")]:
        o, c = content.count(open_ch), content.count(close_ch)
        if o != c:
            problems.append(f"unbalanced '{open_ch}{close_ch}': {o} open vs {c} close")
    return problems


_CLASS_RE = re.compile(r'public\s+(?:sealed\s+|abstract\s+|partial\s+)*(?:class|struct|interface)\s+(\w+)')


def _check_class_name(entry, content):
    if entry["type"] != "csharp":
        return []
    filename_stem = os.path.splitext(os.path.basename(entry["path"]))[0]
    matches = _CLASS_RE.findall(content)
    if len(matches) == 1 and matches[0] != filename_stem:
        return [f"public type '{matches[0]}' doesn't match filename '{filename_stem}' -- "
                f"Unity requires these to match exactly for MonoBehaviours to attach in the Editor"]
    return []


def _validate_file(entry, content):
    return _check_balance(content) + _check_class_name(entry, content)


def _clean_fences(content):
    content_clean = content.strip()
    if content_clean.startswith("```"):
        lines = content_clean.split("\n")
        if lines[0].startswith("```"):
            lines = lines[1:]
        if lines and lines[-1].strip() == "```":
            lines = lines[:-1]
        content_clean = "\n".join(lines)
    return content_clean


def run_generation(file_plan, spec_text, output_dir):
    """Step 2 of the pipeline: write every planned file into output_dir."""

    def normalize_path(entry):
        p = entry["path"].lstrip("/")
        if not (p.startswith("Assets/") or p.startswith("Packages/")):
            p = f"Assets/{p}"
            entry["path"] = p
        return p

    def build_user_prompt(entry, extra_instruction=None):
        spec_excerpt_max_chars = 6000
        spec_for_prompt = spec_text[:spec_excerpt_max_chars]
        if len(spec_text) > spec_excerpt_max_chars:
            spec_for_prompt += ("\n\n[...spec truncated for length -- see the description field "
                                 "below for this file's specific requirements...]")
        prompt = (
            f"Specification excerpt, for background context:\n\n{spec_for_prompt}\n\n"
            f"---\n\nNow write this specific file in full:\n\n"
            f"Path: {entry['path']}\nType: {entry['type']}\nRequired contents: {entry['description']}\n\n"
            f"Write the complete file content now."
        )
        if extra_instruction:
            prompt += f"\n\nIMPORTANT -- fix this specific problem from the previous attempt: {extra_instruction}"
        return prompt

    def write_file(entry, content):
        out_path = os.path.join(output_dir, entry["path"])
        os.makedirs(os.path.dirname(out_path), exist_ok=True)
        content_clean = _clean_fences(content)
        with open(out_path, "w", encoding="utf-8") as f:
            f.write(content_clean)
        _log(f"  -> {entry['path']}: {len(content_clean)} chars written.")
        return content_clean

    generation_report = {}

    def generate_with_retries(entry, base_prompt, initial_content, initial_hit_cap):
        if initial_hit_cap:
            content = generate_long(GENERATION_SYSTEM_PROMPT, base_prompt,
                                     max_new_tokens=MAX_NEW_TOKENS_PER_CALL,
                                     max_continuations=MAX_CONTINUATIONS, temperature=FILE_GEN_TEMPERATURE,
                                     seed_text=initial_content)
        else:
            content = initial_content

        attempts = 1
        issues = _validate_file(entry, _clean_fences(content))
        while issues and attempts <= MAX_FIX_ATTEMPTS:
            _log(f"  {entry['path']}: validation failed ({'; '.join(issues)}) -- "
                 f"retry {attempts}/{MAX_FIX_ATTEMPTS}...")
            retry_prompt = build_user_prompt(entry, extra_instruction="; ".join(issues))
            content = generate_long(GENERATION_SYSTEM_PROMPT, retry_prompt,
                                     max_new_tokens=MAX_NEW_TOKENS_PER_CALL,
                                     max_continuations=MAX_CONTINUATIONS, temperature=FILE_GEN_TEMPERATURE)
            issues = _validate_file(entry, _clean_fences(content))
            attempts += 1

        status = "ok" if attempts == 1 and not issues else ("retried_ok" if not issues else "failed")
        generation_report[entry["path"]] = {"status": status, "attempts": attempts, "issues": issues}
        if issues:
            _log(f"  \u26a0\ufe0f {entry['path']}: still failing validation after {attempts} "
                 f"attempt(s): {'; '.join(issues)}")
        return content

    pending = []
    for entry in file_plan:
        normalize_path(entry)
        out_path = os.path.join(output_dir, entry["path"])
        if os.path.exists(out_path):
            _log(f"Skipping (already generated this run): {entry['path']}")
            generation_report.setdefault(entry["path"], {"status": "ok", "attempts": 0, "issues": []})
        else:
            pending.append(entry)

    _log(f"{len(pending)} file(s) to generate, {len(file_plan) - len(pending)} already done.")

    gen_start_time = time.time()
    batch_start = 0
    while batch_start < len(pending):
        batch = pending[batch_start: batch_start + CURRENT_BATCH_SIZE]
        batch_start += len(batch)
        _log(f"Generating batch of {len(batch)}: {[e['path'] for e in batch]}")

        user_prompts = [build_user_prompt(entry) for entry in batch]
        batch_results = generate_batch(GENERATION_SYSTEM_PROMPT, user_prompts,
                                        max_new_tokens=MAX_NEW_TOKENS_PER_CALL, temperature=FILE_GEN_TEMPERATURE)

        for entry, prompt, (chunk, hit_cap) in zip(batch, user_prompts, batch_results):
            _set_activity(f"Generating {entry['path']}...")
            _file_step_start = time.time()
            content = generate_with_retries(entry, prompt, chunk, hit_cap)
            written = write_file(entry, content)
            _record_step(os.path.basename(entry["path"]), f"{len(written):,} chars",
                         time.time() - _file_step_start, phase="generate")

    gen_elapsed = time.time() - gen_start_time

    failed = [p for p, r in generation_report.items() if r["status"] == "failed"]
    retried = [p for p, r in generation_report.items() if r["status"] == "retried_ok"]
    _log(f"\u2705 All {len(file_plan)} files generated in {output_dir} ({gen_elapsed:.0f}s this run)")
    _log(f"   {len(retried)} needed a retry and passed after fixing; {len(failed)} still flagged "
         f"after {MAX_FIX_ATTEMPTS} attempts.")
    if failed:
        _log(f"   Flagged files: {failed}")

    return generation_report, gen_elapsed


def run_bootstrap(output_dir, unity_version=TARGET_UNITY_VERSION):
    _set_activity("Writing ProjectSettings/ boilerplate...")
    _bootstrap_step_start = time.time()
    project_settings_dir = os.path.join(output_dir, "ProjectSettings")
    os.makedirs(project_settings_dir, exist_ok=True)

    with open(os.path.join(project_settings_dir, "ProjectVersion.txt"), "w") as f:
        f.write(f"m_EditorVersion: {unity_version}\n")
        f.write(f"m_EditorVersionWithRevision: {unity_version} (0000000000)\n")

    editor_build_settings = """%YAML 1.1
%TAG !u! tag:unity3d.com,2011:
--- !u!1045 &1
EditorBuildSettings:
  m_ObjectHideFlags: 0
  serializedVersion: 2
  m_Scenes: []
  m_configObjects: {}
"""
    with open(os.path.join(project_settings_dir, "EditorBuildSettings.asset"), "w") as f:
        f.write(editor_build_settings)

    os.makedirs(os.path.join(output_dir, "Assets"), exist_ok=True)

    assets_path = os.path.join(output_dir, "Assets")
    assets_contents = []
    for root, dirs, files in os.walk(assets_path):
        for fn in files:
            assets_contents.append(os.path.relpath(os.path.join(root, fn), output_dir))

    _log(f"\u2705 Wrote ProjectSettings/ boilerplate (version: {unity_version}). "
         f"Assets/ contains {len(assets_contents)} file(s).")
    if len(assets_contents) == 0:
        _log("\u26a0\ufe0f WARNING: Assets/ is empty -- every planned file landed somewhere else.")

    _record_step("Bootstrap", f"{len(assets_contents)} file(s)",
                 time.time() - _bootstrap_step_start, phase="bootstrap")

    return assets_contents


def run_readme(file_plan, generation_report, output_dir, unity_version=TARGET_UNITY_VERSION):
    _set_activity("Writing README.md...")
    _readme_step_start = time.time()
    by_type = {}
    for entry in file_plan:
        by_type.setdefault(entry["type"], []).append(entry)

    readme_lines = [
        "# Generated Unity Project",
        "",
        f"Generated by: {'Unity-tuned LoRA adapter (vLLM)' if USING_TUNED_MODEL else 'base model (no fine-tuning)'}",
        f"Target Unity version: {unity_version}",
        f"Total files: {len(file_plan)}",
        "",
        "## Files by type",
        "",
    ]
    for ftype, entries in sorted(by_type.items()):
        readme_lines.append(f"### {ftype} ({len(entries)})")
        for entry in entries:
            status = generation_report.get(entry["path"], {}).get("status", "unknown")
            flag = " \u26a0\ufe0f flagged" if status == "failed" else ""
            readme_lines.append(f"- `{entry['path']}`{flag} -- {entry['description']}")
        readme_lines.append("")

    readme_lines += [
        "## Before opening in Unity",
        "",
        "1. See `HUMAN_SETUP.md` (generated alongside this file) for what needs to be wired up "
        "manually in the Editor -- scenes, prefabs, and ScriptableObject import are not auto-generated.",
        "2. Run `Tools > Run Project Setup` from the Editor menu (SetupRunner.cs) to build the "
        "scene and attach the generated scripts automatically.",
        "3. Check `generation_report.json` bundled in this zip's `logs/` folder for any files "
        "that needed a retry or are still flagged after validation.",
    ]

    readme_text = "\n".join(readme_lines)
    with open(os.path.join(output_dir, "README.md"), "w", encoding="utf-8") as f:
        f.write(readme_text)

    _log(f"\u2705 Wrote README.md summarizing {len(file_plan)} files across {len(by_type)} type(s).")
    _record_step("README.md", f"{len(readme_text):,} chars",
                 time.time() - _readme_step_start, phase="readme")


def run_zip(file_plan, output_dir, job_dir):
    _set_activity("Zipping the generated project...")
    _zip_step_start = time.time()
    min_file_bytes = 20
    missing, empty, tiny = [], [], []
    for entry in file_plan:
        fp = os.path.join(output_dir, entry["path"])
        if not os.path.exists(fp):
            missing.append(entry["path"])
            continue
        size = os.path.getsize(fp)
        if size == 0:
            empty.append(entry["path"])
        elif size < min_file_bytes:
            tiny.append((entry["path"], size))

    if missing or empty or tiny:
        _log("\u26a0\ufe0f Integrity check found issues before zipping:")
        if missing:
            _log(f"  Missing entirely ({len(missing)}): {missing}")
        if empty:
            _log(f"  Empty (0 bytes) ({len(empty)}): {empty}")
        if tiny:
            _log(f"  Suspiciously small (<{min_file_bytes}B) ({len(tiny)}): {tiny}")
        _log("  Zipping anyway -- flagged files are also listed in generation_report.json.")
    else:
        _log(f"\u2705 Integrity check passed -- all {len(file_plan)} planned files present, "
             f"non-empty, and reasonably sized.")

    zip_basename = os.path.join(job_dir, "generated_project")
    zip_path = shutil.make_archive(zip_basename, "zip", output_dir)

    final_path = os.path.join(job_dir, "generated_project.zip")
    if zip_path != final_path:
        shutil.move(zip_path, final_path)

    size_mb = os.path.getsize(final_path) / 1e6
    _log(f"\u2705 Zip created: {final_path} ({size_mb:.2f} MB), "
         f"{len(file_plan)} generated files (plus README.md and HUMAN_SETUP.md).")
    _record_step("Zip", f"{size_mb:.2f} MB", time.time() - _zip_step_start, phase="zip")

    return final_path


def run_validation(final_path, output_dir, gen_elapsed, retried, failed):
    _set_activity("Validating generated files (syntax check)...")
    _validation_step_start = time.time()
    ts_available = False
    try:
        from tree_sitter_languages import get_parser
        cs_parser = get_parser("c_sharp")
        ts_available = True
    except Exception as e:
        _log(f"\u26a0\ufe0f tree-sitter unavailable ({e}) -- falling back to structural-only "
             f"validation (brace/paren/bracket balance).")

    def find_ts_errors(tree):
        errors = []
        def walk(node):
            if node.type == "ERROR" or node.is_missing:
                line = node.start_point[0] + 1
                errors.append(f"line {line}: syntax error near '{node.type}'")
            for child in node.children:
                walk(child)
        walk(tree.root_node)
        return errors

    def validate_cs_content(content):
        if ts_available:
            tree = cs_parser.parse(bytes(content, "utf-8"))
            return find_ts_errors(tree)
        return _check_balance(content)

    validation_report = {}
    with zipfile.ZipFile(final_path) as zf:
        names = zf.namelist()
        cs_files = [n for n in names if n.endswith(".cs")]
        json_files = [n for n in names if n.endswith(".json")]

        for n in cs_files:
            content = zf.read(n).decode("utf-8", errors="replace")
            validation_report[n] = {"type": "csharp", "errors": validate_cs_content(content)}

        for n in json_files:
            content = zf.read(n).decode("utf-8", errors="replace")
            try:
                _json.loads(content)
                errors = []
            except Exception as e:
                errors = [str(e)]
            validation_report[n] = {"type": "json", "errors": errors}

    failed_validation = {n: r for n, r in validation_report.items() if r["errors"]}
    total_checked = len(validation_report)

    if failed_validation:
        _log(f"\u274c {len(failed_validation)}/{total_checked} file(s) FAILED validation.")
        for n, r in list(failed_validation.items())[:10]:
            _log(f"  {n} ({r['type']}): {r['errors'][:3]}")
    else:
        _log(f"\u2705 All {total_checked} checked file(s) passed validation with no errors.")

    total_lines, total_chars = 0, 0
    for root, _, files in os.walk(output_dir):
        for fn in files:
            fp = os.path.join(root, fn)
            try:
                with open(fp, "r", encoding="utf-8", errors="ignore") as f:
                    text = f.read()
                total_lines += text.count("\n") + 1
                total_chars += len(text)
            except Exception:
                pass

    pass_rate = 100.0 * (total_checked - len(failed_validation)) / total_checked if total_checked else 100.0
    stats = {
        "model_used": "Unity-tuned adapter (vLLM LoRA)" if USING_TUNED_MODEL else "base model (no tuning)",
        "tensor_parallel_size": TENSOR_PARALLEL_SIZE,
        "total_lines": total_lines,
        "total_chars": total_chars,
        "generation_seconds": round(gen_elapsed, 1),
        "syntax_pass_rate": round(pass_rate, 1),
        "files_retried": len(retried),
        "files_still_flagged": len(failed),
    }
    _log(f"--- Project summary --- {stats}")
    _record_step("Validation", f"{total_checked} file(s) checked",
                 time.time() - _validation_step_start, phase="validate")

    return validation_report, stats


def run_full_pipeline(spec_text, job_dir):
    """The single entry point the job server calls. Runs
    planning -> generation -> bootstrap -> readme -> zip -> validation against an isolated
    per-job directory and returns (final_zip_path, report_dict)."""
    output_dir = os.path.join(job_dir, "generated_project")
    log_dir = os.path.join(job_dir, "logs")
    os.makedirs(output_dir, exist_ok=True)
    os.makedirs(log_dir, exist_ok=True)

    plan_cache_path = os.path.join(job_dir, "file_plan.json")
    if os.path.exists(plan_cache_path):
        with open(plan_cache_path, encoding="utf-8") as f:
            file_plan = _json.load(f)
        _log(f"Resuming job: reusing the file plan from a previous attempt on this job "
             f"({len(file_plan)} files) instead of re-planning.")
    else:
        file_plan = run_planning(spec_text)
        with open(plan_cache_path, "w", encoding="utf-8") as f:
            _json.dump(file_plan, f, indent=2)

    generation_report, gen_elapsed = run_generation(file_plan, spec_text, output_dir)
    run_bootstrap(output_dir)
    run_readme(file_plan, generation_report, output_dir)
    final_path = run_zip(file_plan, output_dir, job_dir)

    retried = [p for p, r in generation_report.items() if r["status"] == "retried_ok"]
    failed = [p for p, r in generation_report.items() if r["status"] == "failed"]
    validation_report, stats = run_validation(final_path, output_dir, gen_elapsed, retried, failed)

    with open(os.path.join(log_dir, "generation_report.json"), "w", encoding="utf-8") as f:
        _json.dump(generation_report, f, indent=2)
    with open(os.path.join(log_dir, "validation_report.json"), "w", encoding="utf-8") as f:
        _json.dump(validation_report, f, indent=2)

    return final_path, {
        "file_plan": file_plan,
        "generation_report": generation_report,
        "validation_report": validation_report,
        "stats": stats,
    }


print("\u2705 Pipeline functions defined: run_planning, run_generation, run_bootstrap, "
      "run_readme, run_zip, run_validation, run_full_pipeline.")


## Step 6 — Repair logic (fix a project's compiler errors)

In [ ]:
# --- Repair logic: given an existing project zip + a compiler-errors report, fix just the -------
# --- files that are broken. Uses the SAME vLLM `llm` engine as generation above. ----------------
import io
from collections import defaultdict

MAX_NEW_TOKENS_PER_FILE = 1536

REPAIR_SYSTEM_PROMPT = (
    "You are an expert Unity/C# engineer fixing compiler errors in an "
    "Editor-tooling project. You are given one file's full current content "
    "and the exact compiler errors that point into it. Return ONLY the "
    "complete corrected file content inside a single ```csharp code block. "
    "No explanation before or after the code block. Preserve every method "
    "signature and class name that isn't directly implicated by an error."
)

CODE_BLOCK_RE = re.compile(r"```(?:csharp|cs)?\s*\n(.*?)```", re.DOTALL)


def build_repair_prompt(file_path, file_content, file_errors, general_instructions):
    err_lines = []
    for e in file_errors:
        hint = e.get("hint", {})
        err_lines.append(
            f"- line {e['line']}, col {e.get('column', '?')}: {e['code']} "
            f"{e['message']}" + (f"  [hint: {hint}]" if hint.get("kind") not in (None, "other") else "")
        )
    errors_block = "\n".join(err_lines)

    return (
        f"{general_instructions}\n\n"
        f"FILE: {file_path}\n\n"
        f"COMPILER ERRORS IN THIS FILE:\n{errors_block}\n\n"
        f"CURRENT FILE CONTENT:\n```csharp\n{file_content}\n```\n\n"
        f"Return the complete corrected file."
    )


def extract_code(model_output):
    m = CODE_BLOCK_RE.search(model_output)
    if m:
        return m.group(1).strip() + "\n"
    return model_output.strip() + "\n"


def generate_fix(file_path, file_content, file_errors, general_instructions):
    """One repair call through vLLM. With multiple broken files, repair_project below hands
    ALL of them to vLLM in a single generate() call (like generate_batch) so its scheduler
    batches them together, running across both tensor-parallel-sharded GPUs at once."""
    user_prompt = build_repair_prompt(file_path, file_content, file_errors, general_instructions)
    messages = [
        {"role": "system", "content": REPAIR_SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt},
    ]
    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    sampling_params = SamplingParams(max_tokens=MAX_NEW_TOKENS_PER_FILE, temperature=0.0, top_p=1.0)
    outputs = llm.generate([prompt], sampling_params, lora_request=ADAPTER_LORA_REQUEST, use_tqdm=False)
    text = outputs[0].outputs[0].text
    return extract_code(text)


def repair_project(zip_bytes, errors_report, general_instructions):
    """Returns (repaired_zip_bytes, summary_dict). Every broken file's fix prompt is submitted
    to vLLM in ONE generate() call -- its own scheduler batches and parallelizes them across the
    tensor-parallel-sharded model, same as generate_batch does for file generation."""
    errors_by_file = defaultdict(list)
    for e in errors_report.get("errors", []):
        errors_by_file[e["file"]].append(e)

    changed, failed = [], []

    src_zip = zipfile.ZipFile(io.BytesIO(zip_bytes))
    names_to_fix = [info.filename for info in src_zip.infolist() if info.filename in errors_by_file]

    prompts, prompt_names, originals = [], [], {}
    for name in names_to_fix:
        try:
            original_text = src_zip.read(name).decode("utf-8")
            originals[name] = original_text
            messages = [
                {"role": "system", "content": REPAIR_SYSTEM_PROMPT},
                {"role": "user", "content": build_repair_prompt(name, original_text, errors_by_file[name],
                                                                  general_instructions)},
            ]
            prompts.append(tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True))
            prompt_names.append(name)
        except Exception as e:
            print(f"[repair_project] FAILED to prepare {name}: {e}")
            failed.append({"file": name, "error": str(e)})

    fixed_results = {}
    if prompts:
        _set_activity(f"Repairing {len(prompts)} broken file(s)...")
        sampling_params = SamplingParams(max_tokens=MAX_NEW_TOKENS_PER_FILE, temperature=0.0, top_p=1.0)
        _repair_batch_start = time.time()
        outputs = llm.generate(prompts, sampling_params, lora_request=ADAPTER_LORA_REQUEST, use_tqdm=False)
        _repair_batch_elapsed = time.time() - _repair_batch_start
        # vLLM fixes every broken file in ONE batched call, so there's no true per-file timing --
        # split the batch's total time evenly across the files as a reasonable approximation.
        _per_file_seconds = _repair_batch_elapsed / len(prompts) if prompts else 0.0
        for name, output in zip(prompt_names, outputs):
            try:
                fixed_results[name] = extract_code(output.outputs[0].text)
                changed.append(name)
                _record_step(os.path.basename(name), f"{len(fixed_results[name]):,} chars",
                             _per_file_seconds, phase="repair")
            except Exception as e:
                print(f"[repair_project] FAILED on {name}: {e}")
                failed.append({"file": name, "error": str(e)})

    out_buf = io.BytesIO()
    out_zip = zipfile.ZipFile(out_buf, "w", zipfile.ZIP_DEFLATED)

    for info in src_zip.infolist():
        name = info.filename
        if name in fixed_results:
            out_zip.writestr(name, fixed_results[name])
        else:
            out_zip.writestr(name, src_zip.read(name))

    out_zip.close()

    summary = {
        "changed": changed,
        "created": [],
        "deleted": [],
        "failed": failed,
        "still_needed_packages": errors_report.get("missingPackages", []),
    }
    return out_buf.getvalue(), summary


print("\u2705 Repair functions defined: generate_fix, repair_project.")


## Step 7 — Unified job server (`/generate` + `/repair`)

In [ ]:
# --- Unified job server: generation AND repair, one queue, one tunnel -------------------
# Jobs are still processed strictly sequentially by ONE worker thread through ONE queue -- that
# keeps the _jobs/_job_queue bookkeeping simple. Within a single job, though, batched generation
# and multi-file repair calls go through vLLM's own scheduler and tensor-parallel engine, so they
# already run across both GPUs simultaneously without any manual thread/lock management here.

import threading, queue, uuid, traceback
from flask import Flask, request, jsonify, send_file

_jobs = {}
_job_queue = queue.Queue()

MAX_JOB_RETRIES = 4
JOB_RETRY_DELAY_S = 10


def _combine_docs_zip(zip_bytes):
    """Extract a spec_docs.zip (a handful of .md/.txt files) into one combined spec string,
    with a sensible file-priority ordering so the most structurally important docs come first."""
    extract_dir = os.path.join(WORKDIR, "spec_docs_job")
    if os.path.isdir(extract_dir):
        shutil.rmtree(extract_dir)
    os.makedirs(extract_dir, exist_ok=True)
    with zipfile.ZipFile(io.BytesIO(zip_bytes)) as zf:
        zf.extractall(extract_dir)
    priority = ["README.md", "Architecture.md", "ScriptsIndex.md", "ProjectVersion.txt",
                "Packages.md", "Scenes.md", "Prefabs.md", "Dependencies.md", "CodingGuidelines.md"]
    files = {f: os.path.join(extract_dir, f) for f in os.listdir(extract_dir)
             if f.endswith((".md", ".txt"))}
    ordered = [f for f in priority if f in files] + sorted(f for f in files if f not in priority)
    parts = []
    for fname in ordered:
        with open(files[fname], encoding="utf-8", errors="ignore") as fh:
            parts.append(f"=== {fname} ===\n\n{fh.read().strip()}")
    return "\n\n\n".join(parts)


def _recover(delay=JOB_RETRY_DELAY_S):
    gc.collect()
    torch.cuda.empty_cache()
    time.sleep(delay)


def _run_generate_job(job_id, spec_text):
    job_dir = os.path.join(JOBS_DIR, job_id)
    os.makedirs(job_dir, exist_ok=True)

    last_err = None
    for attempt in range(1, MAX_JOB_RETRIES + 2):
        if attempt > 1:
            _log(f"[{job_id[:8]}] Auto-retry {attempt - 1}/{MAX_JOB_RETRIES} after a failure -- "
                 f"resuming, not restarting (already-generated files are kept as-is).")
            _jobs[job_id]["retries"] = attempt - 1
            _recover()
        try:
            final_path, _report = run_full_pipeline(spec_text, job_dir)
            job_zip_path = os.path.join(JOBS_DIR, f"{job_id}.zip")
            shutil.copy(final_path, job_zip_path)
            return job_zip_path
        except Exception as e:
            last_err = e
            _log(f"[{job_id[:8]}] Generation failure on attempt {attempt}: {type(e).__name__}: {e}")
            continue

    raise RuntimeError(
        f"Job still failing after {MAX_JOB_RETRIES} auto-retries. Whatever this job already "
        f"generated is saved under {job_dir}."
    ) from last_err


def _run_repair_job(job_id, zip_bytes, errors_report, general_instructions):
    last_err = None
    for attempt in range(1, MAX_JOB_RETRIES + 2):
        if attempt > 1:
            _log(f"[{job_id[:8]}] Auto-retry {attempt - 1}/{MAX_JOB_RETRIES} after a failure -- "
                 f"repair has no per-file checkpoint, so this re-attempts the whole job.")
            _jobs[job_id]["retries"] = attempt - 1
            _recover()
        try:
            repaired_bytes, summary = repair_project(zip_bytes, errors_report, general_instructions)
            job_zip_path = os.path.join(JOBS_DIR, f"{job_id}.zip")
            with open(job_zip_path, "wb") as f:
                f.write(repaired_bytes)
            summary_path = os.path.join(JOBS_DIR, f"{job_id}.summary.json")
            with open(summary_path, "w") as f:
                _json.dump(summary, f)
            return job_zip_path
        except Exception as e:
            last_err = e
            _log(f"[{job_id[:8]}] Repair failure on attempt {attempt}: {type(e).__name__}: {e}")
            continue

    raise RuntimeError(f"Repair job still failing after {MAX_JOB_RETRIES} auto-retries.") from last_err


def _worker_loop():
    global _CURRENT_JOB_ID
    while True:
        job_id, task = _job_queue.get()
        _CURRENT_JOB_ID = job_id
        try:
            _jobs[job_id]["state"] = "running"
            _jobs[job_id]["started_at"] = time.time()
            _set_activity(f"Starting {task['type']} job {job_id[:8]}...")
            if task["type"] == "generate":
                result_path = _run_generate_job(job_id, task["spec_text"])
            else:
                result_path = _run_repair_job(job_id, task["zip_bytes"], task["errors_report"],
                                               task["general_instructions"])
            _jobs[job_id]["state"] = "done"
            _jobs[job_id]["result_path"] = result_path
            _set_activity(f"Job {job_id[:8]} finished -- idle, waiting for the next job.")
        except Exception as e:
            _jobs[job_id]["state"] = "error"
            _jobs[job_id]["error"] = f"{type(e).__name__}: {e}\n{traceback.format_exc()[-2000:]}"
            _set_activity(f"Job {job_id[:8]} failed ({type(e).__name__}) -- idle, waiting for the next job.")
        _jobs[job_id]["finished_at"] = time.time()
        _job_queue.task_done()
        _CURRENT_JOB_ID = None


threading.Thread(target=_worker_loop, daemon=True).start()

app = Flask("unified_server_vllm")

API_KEY = os.environ.get("API_KEY", "dev-only-change-me")
try:
    from kaggle_secrets import UserSecretsClient as _USC
    _secret_key = _USC().get_secret("API_KEY")
    if _secret_key:
        API_KEY = _secret_key
        print("\u2705 API_KEY loaded from Kaggle Secrets.")
except Exception:
    pass
if API_KEY == "dev-only-change-me":
    print("\u26a0\ufe0f No API_KEY secret found -- using the dev fallback key. Set a real one in "
          "Add-ons -> Secrets before exposing this publicly via ngrok.")


def _check_auth():
    auth = request.headers.get("Authorization", "")
    key = auth[7:] if auth.startswith("Bearer ") else request.headers.get("X-API-Key")
    return key == API_KEY


@app.route("/generate", methods=["POST"])
def http_generate():
    if not _check_auth():
        return jsonify({"error": "unauthorized"}), 401
    if "docs" not in request.files:
        return jsonify({"error": "missing 'docs' multipart field (a .zip of spec docs)"}), 400
    spec_text = _combine_docs_zip(request.files["docs"].read())
    job_id = str(uuid.uuid4())
    _jobs[job_id] = {"state": "queued", "type": "generate", "error": None, "result_path": None,
                      "queued_at": time.time(), "started_at": None, "finished_at": None, "retries": 0}
    _job_queue.put((job_id, {"type": "generate", "spec_text": spec_text}))
    return jsonify({"job_id": job_id, "state": "queued"})


@app.route("/repair", methods=["POST"])
def http_repair():
    if not _check_auth():
        return jsonify({"error": "unauthorized"}), 401
    if "project" not in request.files or "errors" not in request.files:
        return jsonify({"error": "missing 'project' and/or 'errors' multipart field"}), 400
    zip_bytes = request.files["project"].read()
    try:
        errors_report = _json.loads(request.files["errors"].read())
    except _json.JSONDecodeError as e:
        return jsonify({"error": f"CompilerErrors.json did not parse: {e}"}), 400
    general_instructions = "Fix the listed compiler errors."
    if "instructions" in request.files:
        general_instructions = request.files["instructions"].read().decode("utf-8", errors="ignore")
    job_id = str(uuid.uuid4())
    _jobs[job_id] = {"state": "queued", "type": "repair", "error": None, "result_path": None,
                      "queued_at": time.time(), "started_at": None, "finished_at": None, "retries": 0}
    _job_queue.put((job_id, {"type": "repair", "zip_bytes": zip_bytes,
                              "errors_report": errors_report,
                              "general_instructions": general_instructions}))
    return jsonify({"job_id": job_id, "state": "queued"})


@app.route("/status/<job_id>", methods=["GET"])
def http_status(job_id):
    if not _check_auth():
        return jsonify({"error": "unauthorized"}), 401
    job = _jobs.get(job_id)
    if not job:
        return jsonify({"error": "unknown job_id"}), 404
    return jsonify({"job_id": job_id, "state": job["state"], "type": job["type"], "error": job["error"]})


@app.route("/result/<job_id>", methods=["GET"])
def http_result(job_id):
    if not _check_auth():
        return jsonify({"error": "unauthorized"}), 401
    job = _jobs.get(job_id)
    if not job or job["state"] != "done":
        return jsonify({"error": "job not done (or unknown)"}), 400
    resp = send_file(job["result_path"], mimetype="application/zip",
                      as_attachment=True, download_name="result.zip")
    summary_path = os.path.join(JOBS_DIR, f"{job_id}.summary.json")
    if job["type"] == "repair" and os.path.exists(summary_path):
        with open(summary_path) as f:
            resp.headers["X-Repair-Summary"] = f.read()
    return resp


@app.route("/health", methods=["GET"])
def http_health():
    return jsonify({"status": "ok"})


_DASHBOARD_HTML = """
<!DOCTYPE html><html><head><title>vLLM Generator/Repair Server</title>
<meta http-equiv="refresh" content="5">
<style>
body { font-family: monospace; background: #111; color: #ddd; padding: 20px; }
h1 { color: #7fd; }
h2 { color: #7fd; font-size: 16px; margin-top: 28px; }
table { border-collapse: collapse; width: 100%; margin-top: 12px; }
td, th { border: 1px solid #444; padding: 6px 10px; text-align: left; font-size: 13px; }
.state-done { color: #7f7; } .state-error { color: #f77; }
.state-running { color: #ff7; } .state-queued { color: #aaf; }
.err { color: #f99; white-space: pre-wrap; font-size: 11px; max-width: 500px; }
.activity { background: #1c2b1c; border: 1px solid #3a5; border-radius: 6px; padding: 10px 14px;
            margin-top: 10px; font-size: 14px; }
.activity b { color: #9f9; }
.dim { color: #888; }
.phase-plan { color: #adf; } .phase-generate { color: #9f9; } .phase-repair { color: #fd7; }
.phase-bootstrap, .phase-readme, .phase-zip, .phase-validate { color: #ccc; }
</style></head><body>
<h1>vLLM Generator / Repair server -- tensor-parallel, __TP_SIZE__ GPU(s)</h1>
<div class="activity">Currently doing: <b>__ACTIVITY_TEXT__</b> <span class="dim">(__ACTIVITY_ELAPSED__ ago)</span></div>
<p>Auto-refreshes every 5s. __JOB_COUNT__ job(s) total.</p>
<h2>Jobs</h2>
<table>
<tr><th>Job ID</th><th>Type</th><th>State</th><th>Retries</th><th>Queued</th><th>Started</th><th>Finished</th><th>Duration</th><th>Error</th></tr>
__ROWS__
</table>
<h2>Work log (most recent first)</h2>
<table>
<tr><th>Time</th><th>Job</th><th>Phase</th><th>Item</th><th>Size</th><th>Duration</th></tr>
__STEP_ROWS__
</table>
</body></html>
"""


def _fmt_time(t):
    return time.strftime("%H:%M:%S", time.localtime(t)) if t else "-"


@app.route("/", methods=["GET"])
def http_dashboard():
    rows = []
    for job_id, job in sorted(_jobs.items(), key=lambda kv: kv[1].get("queued_at") or 0, reverse=True):
        duration = ""
        if job.get("started_at") and job.get("finished_at"):
            duration = f"{job['finished_at'] - job['started_at']:.0f}s"
        elif job.get("started_at"):
            duration = f"{time.time() - job['started_at']:.0f}s (running)"
        err = (job.get("error") or "")[:300]
        rows.append(
            f"<tr><td>{job_id[:8]}</td><td>{job.get('type','?')}</td>"
            f"<td class='state-{job.get('state')}'>{job.get('state')}</td>"
            f"<td>{job.get('retries', 0)}</td>"
            f"<td>{_fmt_time(job.get('queued_at'))}</td>"
            f"<td>{_fmt_time(job.get('started_at'))}</td>"
            f"<td>{_fmt_time(job.get('finished_at'))}</td>"
            f"<td>{duration}</td><td class='err'>{err}</td></tr>"
        )

    step_rows = []
    for step in list(_STEP_LOG):
        step_rows.append(
            f"<tr><td>{_fmt_time(step['ts'])}</td><td>{(step['job_id'] or '-')[:8]}</td>"
            f"<td class='phase-{step['phase']}'>{step['phase']}</td>"
            f"<td>{step['name']}</td><td>{step['size_label']}</td>"
            f"<td>{step['seconds']:.0f}s</td></tr>"
        )

    html = _DASHBOARD_HTML.replace("__JOB_COUNT__", str(len(_jobs)))
    html = html.replace("__TP_SIZE__", str(TENSOR_PARALLEL_SIZE))
    html = html.replace("__ROWS__", "\n".join(rows) or "<tr><td colspan=9>No jobs yet.</td></tr>")
    html = html.replace("__ACTIVITY_TEXT__", _ACTIVITY["text"])
    html = html.replace("__ACTIVITY_ELAPSED__", f"{time.time() - _ACTIVITY['since']:.0f}s")
    html = html.replace("__STEP_ROWS__", "\n".join(step_rows) or "<tr><td colspan=6>No work logged yet.</td></tr>")
    return html


def _run_flask():
    app.run(host="0.0.0.0", port=5001, use_reloader=False)


threading.Thread(target=_run_flask, daemon=True).start()
print("Unified vLLM generate+repair job server starting on port 5001...")


## Step 8 — Expose the server via an ngrok tunnel

This cell blocks (it's a heartbeat loop keeping the tunnel alive) -- that's expected. Leave this
notebook running; your GitHub Actions workflows call the printed URL.


In [ ]:
from pyngrok import ngrok

NGROK_AUTH_TOKEN = os.environ.get("NGROK_AUTH_TOKEN", "")
try:
    from kaggle_secrets import UserSecretsClient as _USC2
    NGROK_AUTH_TOKEN = _USC2().get_secret("NGROK_AUTH_TOKEN")
    if NGROK_AUTH_TOKEN:
        print("\u2705 NGROK_AUTH_TOKEN loaded from Kaggle Secrets.")
except Exception:
    pass

if not NGROK_AUTH_TOKEN:
    raise RuntimeError(
        "NGROK_AUTH_TOKEN not found in Kaggle Secrets (Add-ons -> Secrets). Get one free at "
        "https://dashboard.ngrok.com/get-started/your-authtoken, add it as a secret, then re-run."
    )

ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# With a free static ngrok domain, use it explicitly so the GitHub secret never needs updating:
#   public_url = ngrok.connect(5001, domain="your-static-domain.ngrok-free.app")
public_url = ngrok.connect(5001)
print("=" * 70)
print(f"vLLM tensor-parallel unified server is live at: {public_url}")
print(f"Sharded across {TENSOR_PARALLEL_SIZE} GPU(s).")
print("Set this as NGROK_URL in your GitHub repo's Actions secrets.")
print("=" * 70)

import subprocess, time as _t

# CHANGED: a bare "still up" heartbeat gives no way to tell a genuinely slow job (T4 + eager mode
# + bitsandbytes dequant fallback + LoRA can be legitimately slow token-by-token) apart from an
# actually-stuck one. Since Kaggle only runs one cell at a time, you can't just open a fresh cell
# to check on a running job -- so this cell now prints live diagnostics on its own loop instead of
# a silent heartbeat: which job is running and for how long, live GPU utilization (non-zero =
# actively computing), and the tail of the generation log (concrete progress lines, not just
# Flask access-log noise). No separate monitoring cell needed.

def _gpu_util_line():
    try:
        out = subprocess.run(
            ["nvidia-smi", "--query-gpu=utilization.gpu,memory.used,memory.total",
             "--format=csv,noheader,nounits"],
            capture_output=True, text=True, timeout=5,
        ).stdout.strip().split("\n")
        return " | ".join(
            f"GPU{i}: {u.strip()}% util, {m.strip()}MB used"
            for i, line in enumerate(out) for u, m, _ in [line.split(",")]
        )
    except Exception as e:
        return f"(nvidia-smi failed: {e})"

def _job_status_line():
    if not _jobs:
        return "no jobs yet"
    parts = []
    for jid, job in _jobs.items():
        elapsed = f"{time.time()-job['started_at']:.0f}s" if job.get("started_at") else "-"
        parts.append(f"{jid[:8]}={job['state']} (running {elapsed})")
    return ", ".join(parts)

def _log_tail(n=5):
    try:
        with open(GENERATION_LOG) as f:
            lines = f.readlines()
        return "".join(lines[-n:]).strip() or "(log file empty)"
    except Exception as e:
        return f"(no log file yet: {e})"

try:
    while True:
        _t.sleep(20)
        print(f"\n[{_t.strftime('%H:%M:%S')}] JOB: {_job_status_line()}")
        print(f"    GPU:  {_gpu_util_line()}")
        print(f"    LOG:  {_log_tail()}")
except KeyboardInterrupt:
    print("Stopped.")
